In [1]:
import os
import json
import datetime
import locale
from hashlib import sha256
import random
import polars as pl

In [3]:
source_folder = "json_output_embed"

# Ajouter un id unique

In [4]:
random.seed(26112002)

In [5]:
def hash_content(content):
    words = content.split()
    selected_words = random.sample(words, min(10, len(words)))
    selected_text = ' '.join(selected_words)
    return sha256(selected_text.encode('utf-8')).hexdigest()

In [10]:
for root, dirs, files in os.walk(source_folder):
    json_files = [f for f in files if f.endswith(".json")]
    
    if not json_files:
        continue

    for filename in json_files:
        src_path = os.path.join(root, filename)
        with open(src_path, "r") as f:
            file = json.load(f)

        if file['texte'] == "":
            continue

        file['id'] = hash_content(file['texte'])

        with open(src_path, "w") as f:
            json.dump(file, f, ensure_ascii=False, indent=4)

# Créer la database (parquet)

In [4]:
path_list = []

for root, dirs, files in os.walk(source_folder):
    json_files = [f for f in files if f.endswith(".json")]
    
    if not json_files:
        continue

    for filename in json_files:
        src_path = os.path.join(root, filename)
        path_list.append(src_path)

In [5]:
BATCH_SIZE = 2000
OUTPUT_PARQUET = "corpus/corpus.parquet"
ERRORS_CSV = "erreurs_import.csv"

def normalize_to_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def load_valid_record(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            obj = json.load(f)
    except Exception as e:
        return None, {"file": json_path, "reason": f"json invalide: {e}"}

    record_id = obj.get("id")
    if not record_id:
        return None, {"file": json_path, "reason": "id absente"}

    journaux = normalize_to_list(obj.get("journaux"))
    dates = normalize_to_list(obj.get("dates"))

    if len(journaux) != len(dates):
        return None, {
            "file": json_path,
            "reason": f"longueurs différentes journaux={len(journaux)} dates={len(dates)}"
        }

    record = {
        "id": record_id,
        "texte": obj.get("texte"),
        "journaux": journaux,
        "dates": dates,
        "pays": normalize_to_list(obj.get("pays")),
        "embedding": normalize_to_list(obj.get("embedding")),
        "type": obj.get("type"),
        "pertinence": obj.get("pertinence"),
    }

    return record, None

def to_parquet(fichiers_json):
    dfs = []
    errors = []
    batch_records = []

    total_files = 0

    for json_file in fichiers_json:
        total_files += 1
        record, err = load_valid_record(json_file)

        if err is not None:
            errors.append(err)
            continue

        batch_records.append(record)

        if len(batch_records) >= BATCH_SIZE:
            dfs.append(pl.from_dicts(batch_records))
            batch_records = []

    if batch_records:
        dfs.append(pl.from_dicts(batch_records))

    if not dfs:
        print("Aucun enregistrement valide à écrire.")
        if errors:
            pl.from_dicts(errors).write_csv(ERRORS_CSV)
            print(f"Erreurs écrites dans : {ERRORS_CSV}")
        return

    df = pl.concat(dfs, how="diagonal_relaxed", rechunk=True)
    df = df.unique(subset=["id"], keep="first")
    df.write_parquet(OUTPUT_PARQUET)

    if errors:
        pl.from_dicts(errors).write_csv(ERRORS_CSV)

    print(f"Fichiers lus        : {total_files}")
    print(f"Enregistrements OK  : {df.height}")
    print(f"Erreurs             : {len(errors)}")
    print(f"Parquet écrit       : {OUTPUT_PARQUET}")
    if errors:
        print(f"CSV erreurs         : {ERRORS_CSV}")

# Utilisation :
to_parquet(path_list)

Fichiers lus        : 175171
Enregistrements OK  : 172900
Erreurs             : 2269
Parquet écrit       : corpus/corpus.parquet
CSV erreurs         : erreurs_import.csv


In [2]:
!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 1.1 MB/s  0:00:54m0:00:0100:020m


In [3]:
import polars as pl
import numpy as np


rng = np.random.default_rng(seed=26112002)

# Étape 1 : récupérer n_rows via les métadonnées (très léger, pas de scan des données)
import pyarrow.parquet as pq
n_rows = pq.read_metadata("corpus/corpus_complet.parquet").num_rows

# Étape 2 : tirer les indices
indices = sorted(rng.choice(n_rows, size=1729, replace=False).tolist())

# Étape 3 : filtrer en streaming — jamais tout en RAM
df_sample = (
    pl.scan_parquet("corpus/corpus_complet.parquet")
    .with_row_index("__idx")
    .filter(pl.col("__idx").is_in(indices))
    .drop("__idx")
    .collect(engine="streaming")
)

In [7]:
df_sample.schema

Schema([('id', String),
        ('texte', String),
        ('journaux', List(String)),
        ('dates', List(String)),
        ('pays', List(String)),
        ('embedding', List(Float64)),
        ('type', String),
        ('pertinence', List(String)),
        ('entities', List(Struct({'text': String, 'label': String}))),
        ('entities_org', List(String)),
        ('entities_places', List(String)),
        ('tokens', List(String)),
        ('lemmas', List(String)),
        ('upos', List(String)),
        ('feats', List(String)),
        ('bigrams_noun_adj', List(String)),
        ('trigrams_noun_adj', List(String)),
        ('trigrams_with_verbs', List(String))])

In [9]:
df_sample.write_parquet("corpus/corpus_sample.parquet")